In [0]:
%sql
USE CATALOG de_workspace26;
USE SCHEMA tejpal_shop;

In [0]:
base_path = "/Volumes/de_workspace26/tejpal_shop/tejpal_raw_volume/"

In [0]:
from pyspark.sql.functions import current_timestamp, col


In [0]:
customers_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("includeMetadata", "true") \
    .load(base_path + "customers.csv")

customers_bronze = customers_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

customers_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_bronze_customers")

In [0]:
orders_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(base_path + "orders.csv")

orders_bronze = orders_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

orders_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_bronze_orders")

In [0]:
products_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(base_path + "products.csv")

products_bronze = products_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

products_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.tejpal_shop.tejpal_bronze_products")

In [0]:
display(spark.table("tejpal_bronze_customers"))
display(spark.table("tejpal_bronze_orders"))
display(spark.table("tejpal_bronze_products"))

In [0]:
%sql
ALTER TABLE de_workspace26.tejpal_shop.tejpal_bronze_orders
ALTER COLUMN order_id SET NOT NULL;

In [0]:
%sql
DESCRIBE HISTORY de_workspace26.tejpal_shop.tejpal_bronze_orders;

In [0]:
%sql
DESCRIBE DETAIL de_workspace26.tejpal_shop.tejpal_silver_customers;